In [0]:
# Databricks notebook source
# DBTITLE 1,Configuration - Set paths and parameters

# Source paths - updated to use actual Unity Catalog table locations
sourcePath = "/Volumes/pharma_catalog/bronze/processed_data/consolidated_delta/member/"
goldMemPerBrdgTable = "pharma_catalog.gold.member_person_bridge"
goldMemTable = "pharma_catalog.gold.member"

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Gold Person Bridge Table: {goldMemPerBrdgTable}")
print(f"Gold Member Table: {goldMemTable}")
print("="*60)

In [0]:
# Databricks notebook source
# DBTITLE 1,Helper functions

def path_exists(pathToCheck):
    try:
        return len(dbutils.fs.ls(pathToCheck)) > 0
    except Exception:
        return False

In [0]:
# Databricks notebook source
# DBTITLE 1,Import libraries

from pyspark.sql.functions import date_format, monotonically_increasing_id, sha2, concat_ws, col, row_number
from pyspark.sql.window import Window

In [0]:
# Databricks notebook source
# DBTITLE 1,Define SQL transformations

# SQL to join consolidated member data with person bridge
srcsql = """
WITH consolidateMem1 as (
  SELECT 
    brdg.BISInternalPersonID, brdg.UniqueRecord, mem.ClientID, mem.FileID, mem.LoadDateTime, 
    mem.FileLayoutID, mem.FileLayoutDescription, mem.UniquePersonKey, mem.PlanMemberID, 
    mem.SubscriberID, mem.BeneficiaryID, mem.LastName, mem.FirstName, mem.MiddleInitial, 
    mem.EnrolleeUniqueID, mem.DateofBirth, mem.DeceasedDate, mem.Gender, 
    mem.PermanentAddressLine1, mem.PermanentAddressLine2, mem.PermanentCity, 
    mem.PermanentCounty, mem.PermanentState, mem.PermanentZipCode, 
    mem.MailingAddressLine1, mem.MailingAddressLine2, mem.MailingCity, 
    mem.MailingState, mem.MailingZipCode, mem.MailingCounty, mem.PhoneNumber, 
    mem.Email, mem.MedicaidID, mem.Fax, mem.RaceCode, mem.RaceDataSource, 
    mem.CaretakerFirstName, mem.CaretakerLastName, mem.CaretakerMiddleInitial, 
    mem.EthnicityCode, mem.EthnicityDatasource, mem.SpokenLanguage, 
    mem.SpokenLanguagesourcecode, mem.WrittenLanguageCode, mem.WrittenLanguageSourcecode, 
    mem.OtherLanguage, mem.OtherLanguageSourcecode, mem.USCitizen, 
    mem.AlternateKey1, mem.AlternateKey2, mem.AlternateKey3, mem.AlternateKey4, 
    mem.AlternateKey5, mem.AlternateKey6, mem.AlternateKey7, mem.AlternateKey8, 
    mem.AlternateKey9, mem.AlternateKey10, mem.MaskedMemberID, mem.EnrolleeEducation, 
    mem.EnrolleeEmployment, brdg.PMUP, brdg.IsCurrentPMUP, mem.ProductID
  FROM consolidateMem mem
  INNER JOIN goldMemPerBrdg brdg 
    ON mem.UniqueRecord = brdg.UniqueRecord 
    AND mem.FileLayoutID = brdg.FileLayoutID 
    AND brdg.IsCurrentPMUP = 1
)
SELECT 
  BISInternalPersonID, UniqueRecord, ClientID, FileID, LoadDateTime, FileLayoutID, 
  FileLayoutDescription, UniquePersonKey, PlanMemberID, SubscriberID, BeneficiaryID, 
  LastName, FirstName, MiddleInitial, EnrolleeUniqueID, DateofBirth, DeceasedDate, 
  Gender, PermanentAddressLine1, PermanentAddressLine2, PermanentCity, PermanentCounty, 
  PermanentState, PermanentZipCode, MailingAddressLine1, MailingAddressLine2, 
  MailingCity, MailingState, MailingZipCode, MailingCounty, PhoneNumber, Email, 
  MedicaidID, Fax, RaceCode, RaceDataSource, CaretakerFirstName, CaretakerLastName, 
  CaretakerMiddleInitial, EthnicityCode, EthnicityDatasource, SpokenLanguage, 
  SpokenLanguagesourcecode, WrittenLanguageCode, WrittenLanguageSourcecode, 
  OtherLanguage, OtherLanguageSourcecode, USCitizen, AlternateKey1, AlternateKey2, 
  AlternateKey3, AlternateKey4, AlternateKey5, AlternateKey6, AlternateKey7, 
  AlternateKey8, AlternateKey9, AlternateKey10, MaskedMemberID, EnrolleeEducation, 
  EnrolleeEmployment, PMUP, IsCurrentPMUP, ProductID,
  sha2(concat(
    IfNull(BISInternalPersonID,""), "|", IfNull(UniqueRecord,""), "|", 
    IfNull(ClientID,""), "|", IfNull(CAST(FileID AS STRING),""), "|", 
    IfNull(CAST(LoadDateTime AS STRING),""), "|", IfNull(CAST(FileLayoutID AS STRING),""), "|", 
    IfNull(FileLayoutDescription,""), "|", IfNull(UniquePersonKey,""), "|", 
    IfNull(PlanMemberID,""), "|", IfNull(SubscriberID,""), "|", 
    IfNull(BeneficiaryID,""), "|", IfNull(LastName,""), "|", IfNull(FirstName,""), "|", 
    IfNull(MiddleInitial,""), "|", IfNull(EnrolleeUniqueID,""), "|", 
    IfNull(CAST(DateofBirth AS STRING),""), "|", IfNull(CAST(DeceasedDate AS STRING),""), "|", 
    IfNull(Gender,""), "|", IfNull(PermanentAddressLine1,""), "|", 
    IfNull(PermanentAddressLine2,""), "|", IfNull(PermanentCity,""), "|", 
    IfNull(PermanentCounty,""), "|", IfNull(PermanentState,""), "|", 
    IfNull(PermanentZipCode,""), "|", IfNull(MailingAddressLine1,""), "|", 
    IfNull(MailingAddressLine2,""), "|", IfNull(MailingCity,""), "|", 
    IfNull(MailingState,""), "|", IfNull(MailingZipCode,""), "|", 
    IfNull(MailingCounty,""), "|", IfNull(PhoneNumber,""), "|", IfNull(Email,""), "|", 
    IfNull(MedicaidID,""), "|", IfNull(Fax,""), "|", IfNull(RaceCode,""), "|", 
    IfNull(RaceDataSource,""), "|", IfNull(CaretakerFirstName,""), "|", 
    IfNull(CaretakerLastName,""), "|", IfNull(CaretakerMiddleInitial,""), "|", 
    IfNull(EthnicityCode,""), "|", IfNull(EthnicityDatasource,""), "|", 
    IfNull(SpokenLanguage,""), "|", IfNull(SpokenLanguagesourcecode,""), "|", 
    IfNull(WrittenLanguageCode,""), "|", IfNull(WrittenLanguageSourcecode,""), "|", 
    IfNull(OtherLanguage,""), "|", IfNull(OtherLanguageSourcecode,""), "|", 
    IfNull(USCitizen,""), "|", IfNull(AlternateKey1,""), "|", IfNull(AlternateKey2,""), "|", 
    IfNull(AlternateKey3,""), "|", IfNull(AlternateKey4,""), "|", 
    IfNull(AlternateKey5,""), "|", IfNull(AlternateKey6,""), "|", 
    IfNull(AlternateKey7,""), "|", IfNull(AlternateKey8,""), "|", 
    IfNull(AlternateKey9,""), "|", IfNull(AlternateKey10,""), "|", 
    IfNull(MaskedMemberID,""), "|", IfNull(EnrolleeEducation,""), "|", 
    IfNull(EnrolleeEmployment,""), "|", IfNull(PMUP,""), "|", 
    IfNull(CAST(IsCurrentPMUP AS STRING),""), "|", IfNull(ProductID,"")
  ), 256) AS HashKey 
FROM consolidateMem1
"""

In [0]:
# Databricks notebook source
# DBTITLE 1,Main execution - Build Gold Member table

print(f"\nProcessing Gold Member data...")
print(f"Source: {sourcePath}")
print(f"Person Bridge: {goldMemPerBrdgTable}")

if path_exists(sourcePath):
    # Load consolidated member data
    print("Loading consolidated member data...")
    dfconsolidateMemSrc = spark.read.format("delta").load(sourcePath)
    
    # Load person bridge table
    print("Loading person bridge data...")
    dfgoldMemPerBrdg = spark.table(goldMemPerBrdgTable)
    
    # Add UniqueRecord column to consolidated data
    windowPartition = Window.partitionBy(col("FileId")).orderBy(col("RecordHash").desc())
    dfconsolidateMem = dfconsolidateMemSrc.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfconsolidateMemSrc.columns), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FileID"), col("RowNumber")))
    
    # Create temp views
    dfconsolidateMem.createOrReplaceTempView("consolidateMem")
    dfgoldMemPerBrdg.createOrReplaceTempView("goldMemPerBrdg")
    
    # Join member data with person bridge
    print("Joining member data with person bridge...")
    dfsrc = spark.sql(srcsql)
    
    recordCount = dfsrc.count()
    print(f"Found {recordCount} records to process")
    
    if recordCount > 0:
        # Write to gold table
        print(f"\nWriting to Gold Member table: {goldMemTable}")
        spark.sql(f"DROP TABLE IF EXISTS {goldMemTable}")
        dfsrc.write.format("delta").mode("overwrite").saveAsTable(goldMemTable)
        print("\n✓ Gold Member table created successfully!")
    else:
        print("\n⚠ No records to process - check person bridge table has data")
else:
    print(f"\n✗ Source path does not exist: {sourcePath}")